# 05 - Differential Expression & Pathways

**Goal:** within each major cell type, find genes that change when exposed to nanoplastic (vs control), then translate gene lists into *biology* via pathway enrichment.

**Why per cell type:** a gene change in T cells can be invisible in the whole-sample average if monocytes do the opposite. Splitting by cell type isolates the real signal.

**Pathway enrichment:** a list of 200 gene names means little; enrichment tells you those genes cluster into e.g. 'inflammatory response' or 'oxidative stress', which is what you actually report.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # make the src/ package importable
from src import config as cfg
from src import utils as ut
ut.setup_scanpy()

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("scanpy", sc.__version__)

In [ ]:
import gseapy as gp
adata = ut.load_checkpoint('03_annotated')
exposed = [s for s in cfg.SAMPLES if s != cfg.CONTROL_LABEL]
exposed

## DE: each exposed sample vs control, per cell type
Loops over cell types and conditions, runs a Wilcoxon test, and saves a results table per combination. We collect significant up-regulated genes for enrichment.

In [ ]:
de_results = {}
for ct in adata.obs['cell_type'].unique():
    sub = adata[adata.obs['cell_type'] == ct]
    for cond in exposed:
        mask = sub.obs['sample'].isin([cond, cfg.CONTROL_LABEL])
        s2 = sub[mask].copy()
        if s2.obs['sample'].nunique() < 2 or s2.n_obs < 30:
            continue   # skip combos with too few cells
        sc.tl.rank_genes_groups(s2, 'sample', groups=[cond],
                                reference=cfg.CONTROL_LABEL, method='wilcoxon')
        df = sc.get.rank_genes_groups_df(s2, group=cond)
        de_results[(ct, cond)] = df
        df.to_csv(cfg.TABLE_DIR / f'05_DE_{ct}_{cond}_vs_control.csv'.replace(' ', '_'))
print(f'{len(de_results)} cell-type x condition comparisons computed')

## Pathway enrichment example for one comparison
Wrap this in a loop once you've confirmed it works on one cell type / condition. Uses Enrichr gene-set libraries (GO / KEGG / Reactome).

In [ ]:
key = list(de_results)[0]
df = de_results[key]
sig_up = df[(df['pvals_adj'] < 0.05) & (df['logfoldchanges'] > 1)]['names'].tolist()
print(key, '->', len(sig_up), 'up-regulated genes')

if len(sig_up) >= 10:
    enr = gp.enrichr(gene_list=sig_up,
                     gene_sets=['GO_Biological_Process_2021', 'KEGG_2021_Human'],
                     outdir=None)
    display(enr.results.sort_values('Adjusted P-value').head(15))